# BKT: Faithful and Degenerate-Filtered Baselines

The first model. Correctness-only Bayesian Knowledge Tracing on MathDial,
the foundation the misconception channel later builds on.

**What this notebook does:**
1. Loads the annotated MathDial data and flattens it to per-(dialogue, turn, kc).
2. Fits BKT in two conditions:
   - **Faithful**: all observed KCs, including the 19 degenerate (no-variation)
     KCs. These are un-estimable, so they fall back to the weighted-average
     parameters; this mirrors the original pipeline's KC set.
   - **Cleaned (degenerate-filtered)**: KCs with no label variation removed
     before fitting. This is the foundation the misconception channel builds on.
3. For each, builds the weighted-average parameter fallback for unseen KCs,
   predicts on test, and reports Acc / AUC / log-likelihood / Brier against
   the majority-class floor.

BKT is implemented in pure Python / numpy in `extension/myext/bkt.py` (no
compiled dependency), fit per KC by EM with random restarts and the standard
identifiability constraint (guess, slip < 0.5). All logic lives in
`extension/myext`; this notebook only orchestrates and reports. Only the
degenerate filter is applied here; other thresholds are a later sensitivity
sweep.


## Setup

In [1]:
import os
from pathlib import Path

# Walk up to the repo root (folder containing data/annotated) so relative
# paths resolve regardless of where the notebook was opened.
_here = Path.cwd()
for _c in [_here, *_here.parents]:
    if (_c / "data" / "annotated").exists():
        os.chdir(_c)
        break
else:
    raise FileNotFoundError("Could not find data/annotated above " + str(_here))

import sys
# make the extension package importable
sys.path.insert(0, str(Path.cwd() / "extension"))
print("working directory:", os.getcwd())


working directory: /Users/tandon.utsav2/Desktop/Experiment_1


In [2]:
import json
from ast import literal_eval
import numpy as np
import pandas as pd

from myext import filtering, bkt

DATA = Path("data/annotated")
CONV = {c: literal_eval for c in ["annotation"]}

def load(split):
    df = pd.read_csv(DATA / f"mathdial_{split}_atc.csv", converters=CONV)
    # Remove dialogues whose GPT-4o annotation failed (no usable labelled
    # turns), matching the original work. Reports how many were dropped.
    df, n_removed = filtering.drop_failed_annotations(df)
    print(f"{split}: removed {n_removed} failed-annotation dialogues "
          f"-> {len(df)} dialogues")
    return df

def turn_table(df):
    rows = []
    for didx, ann in zip(df.index, df["annotation"]):
        if not isinstance(ann, dict):
            continue
        for tk, info in ann.items():
            if not isinstance(info, dict) or "correct" not in info or "kcs" not in info:
                continue
            c = 1 if info["correct"] in (True, 1, "true", "correct") else 0
            for kc in info["kcs"]:
                rows.append((didx, tk, c, kc))
    return pd.DataFrame(rows, columns=["dialogue_idx", "turn", "correct", "kc"])

train_long = turn_table(load("train"))
test_long = turn_table(load("test"))
print("train obs:", len(train_long), "| test obs:", len(test_long))


train: removed 18 failed-annotation dialogues -> 2235 dialogues
test: removed 7 failed-annotation dialogues -> 588 dialogues
train obs: 29478 | test obs: 7446


In [3]:
# Majority-class floor (the number any model must beat), per turn.
def majority_floor(long_df):
    per_turn = long_df.groupby(["dialogue_idx", "turn"])["correct"].first()
    p = per_turn.mean()
    return max(p, 1 - p), p

test_floor, test_p = majority_floor(test_long)
print(f"test fraction correct (per turn): {test_p:.3f}")
print(f"test majority-class accuracy floor: {test_floor:.3f}")


test fraction correct (per turn): 0.412
test majority-class accuracy floor: 0.588


## Condition 1: Faithful baseline (all observed KCs)

Fits on every observed KC, including the 19 degenerate ones. Unlike the
original compiled pipeline (which crashed on these), the pure-Python EM fits
them to extreme but valid parameters at the identifiability bounds (an
all-correct KC, for instance, fits to prior near 1 and slip near 0, meaning
'students always know this'). So the faithful baseline genuinely runs. We keep
it as the reference reproduction of the original KC set.

Note: fitting is pure-Python EM over all KCs and takes a couple of minutes.


In [4]:
# Faithful: no filtering. The degenerate KCs yield NaN params and fall back
# per-parameter; fit_bkt reports how many.
fitted_faithful = bkt.fit_bkt(train_long, n_restarts=3)
overall_f, final_f, _ = bkt.evaluate(fitted_faithful, test_long)
print("Faithful  overall :", overall_f)
print("Faithful  final   :", final_f)


Faithful  overall : n=7446  Acc=0.6138  AUC=0.6276  LL=-0.6517  Brier=0.2291
Faithful  final   : n=588  Acc=0.4082  AUC=0.5092  LL=-0.7482  Brier=0.2764


## Condition 2: Cleaned baseline (degenerate KCs filtered)

Removes KCs with no label variation in train (the primary, model-theoretic
filter). The keep-set is decided on train only and applied to both splits;
test turns whose KC was filtered or never seen in train fall through to the
weighted-average parameter fallback at prediction time.


In [5]:
train_f, test_f, fres = filtering.filter_splits(
    train_long, test_long, drop_no_variation=True, min_count=1
)
print(fres.summary())
print()
# How many test turns end up on the fallback (filtered-or-unseen KCs)?
kept = fres.keep_kcs
test_unseen_obs = test_long[~test_long["kc"].isin(kept)]
print(f"test observations routed to fallback: {len(test_unseen_obs)} "
      f"of {len(test_long)} ({100*len(test_unseen_obs)/len(test_long):.2f}%)")


KCs: 143 -> 124 (dropped 19)
observations: 29478 -> 29447 (removed 31, 0.11%)
drop reasons: no_variation=19

test observations routed to fallback: 33 of 7446 (0.44%)


In [6]:
# Fit on the filtered train; evaluate on the FULL test (so fallback turns are
# scored too, the honest measure). predict_long handles unseen KCs via fallback.
import time
_t0 = time.time()
fitted_clean = bkt.fit_bkt(train_f, n_restarts=3)
print(f"(fit took {time.time()-_t0:.0f}s)")
overall_c, final_c, preds_c = bkt.evaluate(fitted_clean, test_long)
print("Cleaned  overall :", overall_c)
print("Cleaned  final   :", final_c)
print()
print(f"(majority-class floor on test: {test_floor:.3f})")


(fit took 68s)
Cleaned  overall : n=7446  Acc=0.6135  AUC=0.6271  LL=-0.6503  Brier=0.2291
Cleaned  final   : n=588  Acc=0.4099  AUC=0.5077  LL=-0.7487  Brier=0.2766

(majority-class floor on test: 0.588)


## Fitted parameters and the fallback

Inspect where the parameters landed. The weighted-average fallback is what an
unseen KC defaults to. Worth checking the spread now: if the parameters are
heavily skewed (e.g. many slips near 0), that is the signal to revisit
arithmetic vs log-odds averaging.


In [7]:
params_df = pd.DataFrame(fitted_clean.per_skill).T[list(bkt.PARAM_NAMES)]
print("fitted parameter summary (across KCs):")
print(params_df.describe().round(3).to_string())
print()
print("weighted-average fallback (unseen-KC defaults):")
for k, v in fitted_clean.fallback.items():
    print(f"  {k:8s} {v:.4f}")


fitted parameter summary (across KCs):
         prior   learns  guesses    slips
count  124.000  124.000  124.000  124.000
mean     0.328    0.280    0.217    0.249
std      0.270    0.329    0.172    0.191
min      0.001    0.001    0.001    0.001
25%      0.070    0.001    0.005    0.008
50%      0.305    0.118    0.209    0.275
75%      0.503    0.575    0.359    0.432
max      0.999    0.999    0.490    0.490

weighted-average fallback (unseen-KC defaults):
  prior    0.2260
  learns   0.1616
  guesses  0.2866
  slips    0.2433


## Results summary

Collect the headline numbers. The cleaned baseline is the foundation the
misconception channel builds on; the faithful baseline (if it fit) is the
reference reproduction. Both should clear the majority-class floor on AUC
(accuracy may sit near the floor given class imbalance, which is why AUC is
the headline metric).


In [8]:
rows = [
    ["Faithful (all KCs)", overall_f.accuracy, overall_f.auc,
     overall_f.log_likelihood, overall_f.brier],
    ["Cleaned (degenerate-filtered)", overall_c.accuracy, overall_c.auc,
     overall_c.log_likelihood, overall_c.brier],
]
summary = pd.DataFrame(rows, columns=["condition", "Acc", "AUC", "LogLik", "Brier"])
summary["maj-class floor (Acc)"] = test_floor
print(summary.round(4).to_string(index=False))


                    condition    Acc    AUC  LogLik  Brier  maj-class floor (Acc)
           Faithful (all KCs) 0.6138 0.6276 -0.6517 0.2291                 0.5882
Cleaned (degenerate-filtered) 0.6135 0.6271 -0.6503 0.2291                 0.5882


### Notes

Fill in after running:
- Cleaned AUC vs the majority-class floor: is the model doing real work?
  (AUC well above 0.5 means yes; accuracy may sit near the floor given the
  class imbalance, which is why AUC is the headline metric.)
- Faithful vs cleaned gap: a negligible gap confirms the degenerate KCs were
  immaterial (they fall back anyway), supporting the filtering choice.
- Final-turn AUC is typically near chance, since the final turn is the hardest
  single-turn prediction with the least remaining signal; reporting it
  alongside overall is the convention.
- Parameter spread: check the fitted guesses/slips. Values clustered near the
  0.49 identifiability ceiling indicate near-unidentifiable KCs (correctness
  nearly independent of mastery). Heavy skew is the signal to revisit
  arithmetic vs log-odds averaging for the fallback.
- Identifiability constraint: guess and slip are constrained below 0.5 by
  design (standard BKT), so 'mastered' always means 'more likely correct'.
  This is a modelling choice to document, not just an implementation detail.
